In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

df = pd.read_csv('car_fuel_efficiency_2026.csv')

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (10000, 11)


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [ ]:
df = df.fillna(0)

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

target = 'fuel_efficiency_mpg'

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

train_dicts = df_train.drop(columns=[target]).to_dict(orient='records')
val_dicts = df_val.drop(columns=[target]).to_dict(orient='records')
test_dicts = df_test.drop(columns=[target]).to_dict(orient='records')

dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)
print("Total features:", len(dv.feature_names_))

Training: (6000, 16)
Validation: (2000, 16)
Testing: (2000, 16)
Total features: 16


In [4]:
print(f"Training rows: {len(df_train)}")
print(f"Validation rows: {len(df_val)}")
print(f"Testing rows: {len(df_test)}")

Training rows: 6000
Validation rows: 2000
Testing rows: 2000


### Question 1

In [ ]:
from sklearn.tree import DecisionTreeRegressor, export_text

model = DecisionTreeRegressor(max_depth=1, random_state=1)
model.fit(X_train, y_train)

feature_index = model.tree_.feature[0]
feature_name = dv.get_feature_names_out()[feature_index]

original_feature = feature_name.split('=')[0]

print("Splitting feature:", feature_name)
print("Original feature:", original_feature)
print(export_text(model, feature_names=list(dv.get_feature_names_out())))

Splitting feature: model_year
Original feature: model_year
|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



#### Answer: The decision tree splits at model_year = 1997.5:

### Question 2

In [6]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 1.8371


#### Answer Final answer: 1.837

### Question 3

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

results = {}

for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    results[n] = round(rmse, 3)

    print(f"n_estimators={n}: RMSE={rmse:.3f}")

best_n = min(results, key=results.get)

print(f"\nBest n_estimators: {best_n}")
print(f"Lowest validation RMSE: {results[best_n]:.3f}")

n_estimators=10: RMSE=1.837
n_estimators=50: RMSE=1.772
n_estimators=100: RMSE=1.768
n_estimators=150: RMSE=1.769

Best n_estimators: 100
Lowest validation RMSE: 1.768


#### Answer 3: Option C — 100

### Question 4

In [9]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
import pandas as pd

depths = [10, 15, 20, 25]
n_estimators_values = [10, 50, 100, 150]

results = []

for depth in depths:
    rmse_scores = []

    for n in n_estimators_values:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, y_pred))

        rmse_scores.append(rmse)

        print(f"max_depth={depth}, n_estimators={n}, RMSE={rmse:.4f}")

    mean_rmse = np.mean(rmse_scores)

    results.append({
        "max_depth": depth,
        "mean_RMSE": mean_rmse
    })
results_df = pd.DataFrame(results)

print("\nMean RMSE for each max_depth:")
print(results_df.round(4).to_string(index=False))
best_depth = results_df.loc[
    results_df["mean_RMSE"].idxmin(), "max_depth"
]

print(f"\nBest max_depth: {best_depth}")

max_depth=10, n_estimators=10, RMSE=1.7837
max_depth=10, n_estimators=50, RMSE=1.7486
max_depth=10, n_estimators=100, RMSE=1.7448
max_depth=10, n_estimators=150, RMSE=1.7445
max_depth=15, n_estimators=10, RMSE=1.8299
max_depth=15, n_estimators=50, RMSE=1.7711
max_depth=15, n_estimators=100, RMSE=1.7679
max_depth=15, n_estimators=150, RMSE=1.7669
max_depth=20, n_estimators=10, RMSE=1.8410
max_depth=20, n_estimators=50, RMSE=1.7731
max_depth=20, n_estimators=100, RMSE=1.7692
max_depth=20, n_estimators=150, RMSE=1.7697
max_depth=25, n_estimators=10, RMSE=1.8368
max_depth=25, n_estimators=50, RMSE=1.7721
max_depth=25, n_estimators=100, RMSE=1.7684
max_depth=25, n_estimators=150, RMSE=1.7680

Mean RMSE for each max_depth:
 max_depth  mean_RMSE
        10     1.7554
        15     1.7840
        20     1.7882
        25     1.7863

Best max_depth: 10


#### Answer 4: Option A — 10

In [10]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)
feature_importance = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_
})
feature_importance = feature_importance.sort_values(
    by='importance',
    ascending=False
)

print("All feature importances:")
print(feature_importance.to_string(index=False))
options = [
    'vehicle_weight',
    'horsepower',
    'acceleration',
    'engine_displacement'
]

selected = feature_importance[
    feature_importance['feature'].isin(options)
]

print("\nQuestion 5 - Feature Importance Comparison:")
print(selected.to_string(index=False))
best_feature = selected.iloc[0]['feature']

print(f"\nCorrect Answer: {best_feature}")

All feature importances:
                     feature  importance
                  model_year    0.296001
              vehicle_weight    0.203879
          fuel_type=Gasoline    0.160712
                  horsepower    0.078017
         engine_displacement    0.061005
                acceleration    0.055201
                  origin=USA    0.042144
drivetrain=Front-wheel drive    0.020145
                   num_doors    0.019415
            fuel_type=Diesel    0.017038
            fuel_type=Hybrid    0.015800
  drivetrain=All-wheel drive    0.007682
               num_cylinders    0.007284
                 origin=Asia    0.006391
 drivetrain=Rear-wheel drive    0.005781
               origin=Europe    0.003505

Question 5 - Feature Importance Comparison:
            feature  importance
     vehicle_weight    0.203879
         horsepower    0.078017
engine_displacement    0.061005
       acceleration    0.055201

Correct Answer: vehicle_weight


#### Answer 5: Option A — vehicle_weight

### Question 6

In [13]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [15]:
import xgboost as xgb
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)

watchlist = [
    (dtrain, 'train'),
    (dval, 'val')
]

In [16]:
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

results = {}

for eta in [0.3, 0.1]:

    xgb_params['eta'] = eta

    evals_result = {}

    model = xgb.train(
        params=xgb_params,
        dtrain=dtrain,
        num_boost_round=100,
        evals=watchlist,
        evals_result=evals_result,
        verbose_eval=False
    )
    rmse = evals_result['val']['rmse'][-1]

    results[eta] = rmse

    print(f"eta={eta}: Validation RMSE={rmse:.4f}")
best_eta = min(results, key=results.get)

print("\n-------------------------")
print(f"Best eta: {best_eta}")
print(f"Lowest validation RMSE: {results[best_eta]:.4f}")

eta=0.3: Validation RMSE=1.8265
eta=0.1: Validation RMSE=1.7248

-------------------------
Best eta: 0.1
Lowest validation RMSE: 1.7248


#### Answer 6: Option B — 0.1